# Notebook 5 – Data Validation
**File:** `05_Data_Validation.ipynb`

## Learning objectives
Create checks that detect invalid, inconsistent, or impossible values before analysis or machine learning.

**Why preprocessing is required:** A dataset can have no missing values and no duplicates yet still be wrong. Validation catches values that violate allowed ranges, formats, categories, relationships, or business rules.

- **Simple example:** Age must not be negative.
- **Real-world example:** A hospital patient's temperature must be within a plausible range and use the correct unit.
- **Business example:** An order quantity must be positive and a discount must be between 0 and 100%.
- **AI/ML use case:** Invalid labels or impossible feature values can teach a model incorrect patterns and reduce prediction quality.

In [1]:
import pandas as pd
import numpy as np

data = pd.DataFrame({
    "EmployeeID": [101, 102, 103, 104, 105],
    "Age": [25, -4, 32, 150, 28],
    "Salary": [45000, 52000, -1000, 60000, 48000],
    "Percentage": [85, 105, 70, -2, 92],
    "Gender": ["Female", "Male", "Other", "Unknown", "Female"],
    "JoinDate": ["2022-01-10", "2021-05-01", "2023-03-15", "2019-08-20", "not-a-date"],
    "BirthDate": ["2000-04-01", "1998-07-10", "1995-02-12", "2001-01-01", "1999-09-09"],
    "Department": ["IT", "HR", "Sales", "Finance", "IT"]
})
display(data)

,EmployeeID,Age,Salary,Percentage,Gender,JoinDate,BirthDate,Department
0,101,25,45000,85,Female,2022-01-10,2000-04-01,IT
1,102,-4,52000,105,Male,2021-05-01,1998-07-10,HR
2,103,32,-1000,70,Other,2023-03-15,1995-02-12,Sales
3,104,150,60000,-2,Unknown,2019-08-20,2001-01-01,Finance
4,105,28,48000,92,Female,not-a-date,1999-09-09,IT


## 1. What is data validation?
Data validation checks whether data follows expected rules. Validation usually flags or quarantines bad records; it should not silently change them without an approved correction rule.

A useful validation result contains:
- The row or record identifier.
- The field and rule that failed.
- The invalid value.
- A clear reason.
- A decision: reject, quarantine, correct from a trusted source, or accept with a warning.

## 2. Range validation
A value must be within an allowed minimum and maximum. Examples: age cannot be negative; percentage must be from 0 to 100.

In [3]:
age_invalid = data[(data["Age"] < 0) | (data["Age"] > 120)]
percentage_invalid = data[~data["Percentage"].between(0, 100)]
salary_invalid = data[data["Salary"] < 0]

print("Invalid ages:")
display(age_invalid[["EmployeeID", "Age"]])
print("Invalid percentages:")
display(percentage_invalid[["EmployeeID", "Percentage"]])
print("Invalid salaries:")
display(salary_invalid[["EmployeeID", "Salary"]])

Invalid ages:


,EmployeeID,Age
1,102,-4
3,104,150


Invalid percentages:


,EmployeeID,Percentage
1,102,105
3,104,-2


Invalid salaries:


,EmployeeID,Salary
2,103,-1000


- **Use when:** Numeric fields have known domain limits.
- **Do not use when:** The limits are arbitrary or do not account for the business context.
- **Advantages:** Simple and easy to explain.
- **Limitations:** Plausible ranges vary by population and domain; an out-of-range value may be a unit or data-entry issue rather than a value to delete.

## 3. Type validation
Check whether fields have the expected data type. Convert only when conversion is safe; preserve the original value for investigation.

In [4]:
# Numeric conversion: invalid strings become NaN and can be flagged
raw_salary = pd.Series(["45000", "52000", "unknown", "-1000"])
salary_numeric = pd.to_numeric(raw_salary, errors="coerce")
type_errors = raw_salary[salary_numeric.isna()]
print("Values that could not be converted:")
display(type_errors)

print("Converted values:")
display(salary_numeric)

Values that could not be converted:


2    unknown
dtype: str

Converted values:


0    45000.0
1    52000.0
2        NaN
3    -1000.0
dtype: float64

- **Use when:** Importing CSV/Excel/API data where columns may be inferred incorrectly.
- **Do not use when:** A value is a code or identifier that should remain text (for example, postal codes with leading zeros).
- **Advantages:** Prevents calculation and comparison errors.
- **Limitations:** Automatic conversion can lose formatting or turn invalid text into missing values; inspect conversion failures.

## 4. Format validation
Check strings against expected patterns, such as email addresses, phone numbers, postal codes, or IDs. Regex checks are useful for structure, not proof that an address or phone number is real.

In [5]:
emails = pd.Series(["bilal@example.com", "not-an-email", "user.name@company.in", None])
email_pattern = r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$"
email_valid = emails.str.match(email_pattern, na=False)
display(pd.DataFrame({"email": emails, "is_valid_format": email_valid}))

,email,is_valid_format
0,bilal@example.com,True
1,not-an-email,False
2,user.name@company.in,True
3,NaN,False


- **Use when:** A field has a documented format.
- **Do not use when:** The format is not standardized or regex would reject valid regional formats.
- **Advantages:** Quickly catches common input mistakes.
- **Limitations:** A string can match a pattern but still be fake, unreachable, or unregistered.

## 5. Category validation
Ensure categorical values belong to an approved set. Normalize whitespace/case only if the business rules allow it.

In [6]:
allowed_gender = {"Female", "Male", "Other"}
gender_clean = data["Gender"].astype("string").str.strip()
invalid_gender = ~gender_clean.isin(allowed_gender)
display(data.loc[invalid_gender, ["EmployeeID", "Gender"]])

,EmployeeID,Gender
3,104,Unknown


- **Use when:** Categories come from a controlled list.
- **Do not use when:** The category set is genuinely open-ended or the business allows additional values.
- **Advantages:** Catches spelling differences and unexpected labels.
- **Limitations:** Hard-coded lists can become outdated; do not force a value into a category without evidence.

## 6. Null validation
Some fields are mandatory; others may be optional. Validate required fields separately from optional fields.

In [7]:
required_columns = ["EmployeeID", "Age", "Salary", "Department"]
null_counts = data[required_columns].isna().sum()
print(null_counts)

rows_missing_required = data[data[required_columns].isna().any(axis=1)]
display(rows_missing_required)

EmployeeID    0
Age           0
Salary        0
Department    0
dtype: int64


,EmployeeID,Age,Salary,Percentage,Gender,JoinDate,BirthDate,Department


- **Use when:** A field must be present for the record to be meaningful or legally/process-wise complete.
- **Do not use when:** Missing is a valid state (for example, `EndDate` for a current employee).
- **Advantages:** Protects downstream processes from incomplete records.
- **Limitations:** Treating every null as an error can reject valid records.

## 7. Business rule validation
Business rules describe relationships that are meaningful to the organization. Example: salary must be non-negative; joining date must not precede birth date.

In [ ]:
dates = data.copy()
dates["JoinDate"] = pd.to_datetime(dates["JoinDate"], errors="coerce")
dates["BirthDate"] = pd.to_datetime(dates["BirthDate"], errors="coerce")

invalid_join_before_birth = (
    dates["JoinDate"].notna()
    & dates["BirthDate"].notna()
    & (dates["JoinDate"] < dates["BirthDate"])
)
display(dates.loc[invalid_join_before_birth, ["EmployeeID", "BirthDate", "JoinDate"]])

# More examples of business rules
rule_results = pd.DataFrame({
    "EmployeeID": data["EmployeeID"],
    "age_non_negative": data["Age"] >= 0,
    "salary_non_negative": data["Salary"] >= 0,
    "percentage_0_to_100": data["Percentage"].between(0, 100)
})
display(rule_results)

- **Use when:** The organization has documented rules across fields or processes.
- **Do not use when:** The rule is merely an assumption and has not been confirmed with the domain owner.
- **Advantages:** Detects logically inconsistent records that simple type/range checks miss.
- **Limitations:** Rules need maintenance and can become invalid when policies change.

## 8. Referential validation
A foreign-key value should exist in the referenced table. For example, every order's `CustomerID` should exist in the customer master.

In [ ]:
customers = pd.DataFrame({"CustomerID": [1, 2, 3], "Name": ["Asha", "Ravi", "Mina"]})
orders = pd.DataFrame({"OrderID": [501, 502, 503], "CustomerID": [1, 4, 2]})

invalid_customer_refs = orders[~orders["CustomerID"].isin(customers["CustomerID"])]
display(invalid_customer_refs)

- **Use when:** Tables are related through primary/foreign keys.
- **Do not use when:** The referenced entity is allowed to arrive later in an asynchronous pipeline; use a pending/unmatched status instead.
- **Advantages:** Prevents orphan records and broken joins.
- **Limitations:** Requires the correct reference table and timing; stale master data can produce false failures.

## 9. Date validation
Check parsing, allowed date ranges, and relationships between dates. Use `errors='coerce'` to identify unparseable values, not to silently discard them.

In [ ]:
date_series = pd.Series(["2026-01-15", "15/02/2026", "not-a-date", "2027-12-01"])
parsed_dates = pd.to_datetime(date_series, errors="coerce", dayfirst=True)
date_validation = pd.DataFrame({
    "raw_date": date_series,
    "parsed_date": parsed_dates,
    "is_parseable": parsed_dates.notna()
})
display(date_validation)

- **Use when:** Dates drive reporting, eligibility, ordering, or time-based models.
- **Do not use when:** The source uses an ambiguous date format that has not been clarified.
- **Advantages:** Finds impossible or unparseable date values.
- **Limitations:** Locale ambiguity (for example, `01/02/2026`) can silently swap day and month.

## 10. Constraint validation
Constraints include uniqueness, allowed values, non-null rules, and combinations of columns. Examples: employee ID must be unique; quantity must be positive.

In [ ]:
# Uniqueness constraint
duplicate_employee_ids = data[data.duplicated(subset=["EmployeeID"], keep=False)]
print("Duplicate EmployeeIDs:")
display(duplicate_employee_ids)

# Positive quantity example
transactions = pd.DataFrame({"TransactionID": [1, 2, 3], "Quantity": [2, 0, -1]})
invalid_quantity = transactions[transactions["Quantity"] <= 0]
display(invalid_quantity)

## 11. Build a reusable validation report
Collect failed rules into one table. This example checks the employee data and reports row IDs and reasons.

In [ ]:
issues = []

def add_issue(mask, column, rule_name, reason):
    for idx in data.index[mask.fillna(False)]:
        issues.append({
            "row_index": int(idx),
            "EmployeeID": data.loc[idx, "EmployeeID"],
            "column": column,
            "rule": rule_name,
            "value": data.loc[idx, column],
            "reason": reason
        })

add_issue((data["Age"] < 0) | (data["Age"] > 120), "Age",
          "age_range", "Age must be between 0 and 120")
add_issue(data["Salary"] < 0, "Salary",
          "salary_non_negative", "Salary cannot be negative")
add_issue(~data["Percentage"].between(0, 100), "Percentage",
          "percentage_range", "Percentage must be between 0 and 100")
add_issue(~data["Gender"].isin(allowed_gender), "Gender",
          "accepted_category", "Gender must be in the approved category set")

validation_report = pd.DataFrame(issues)
display(validation_report)

## 12. What should happen after a validation failure?
Do not automatically delete or overwrite invalid records. Depending on the rule:
- **Reject:** Do not accept a record that cannot be used safely.
- **Quarantine:** Store it separately for review.
- **Correct:** Use an authoritative source or confirmed business rule.
- **Warn:** Keep the record but flag a non-critical issue.
- **Escalate:** Ask the data owner when the rule or source is unclear.

## 13. AI/ML workflow example
Validate raw data before splitting/training. Apply learned preprocessing (such as imputers or encoders) inside a pipeline fitted only on training data. Keep validation rules separate from statistical transformations: validation detects unacceptable values; preprocessing transforms acceptable data for analysis/modeling.

## 14. Final checklist
1. Confirm schema and expected types.
2. Check required fields and null rules.
3. Check numeric ranges and category sets.
4. Validate formats and dates.
5. Check business relationships and foreign keys.
6. Check uniqueness/constraints.
7. Produce an issue report with row IDs and reasons.
8. Correct only with evidence; preserve an audit trail.